# Self-Supervised Vision Foundations: DINOv2, Patch L2-Normalization, and Sinkhorn-Knopp

> **Topic**: Vision Transformers, Non-Contrastive SSL, Optimal Transport Clustering  
> **Key Breakthrough**: Extracting universal, camera-invariant spatial representations without manual annotations.

---

## 1. Non-Contrastive Knowledge Distillation
DINOv2 trains a Student network $g_{\theta_s}$ to match predictions of an exponential moving average (EMA) Teacher network $g_{\theta_t}$.

To prevent representation collapse (where all images map to a constant trivial vector) without requiring negative pairs:
1. **Centering & Sharpening**:
   $$P_t(x)^{(i)} = \frac{\exp\left( (g_{\theta_t}(x)^{(i)} - c_i) / \tau_t \right)}{\sum_j \exp\left( (g_{\theta_t}(x)^{(j)} - c_j) / \tau_t \right)}$$
2. **Sinkhorn-Knopp Optimal Transport Algorithm**:
   Assigns tokens uniformly across prototype clusters to guarantee maximum entropy.

### Patch-Level L2 Normalization
As proven in our Kaggle breakthroughs, raw ViT embeddings carry camera sensor gain, white-balance, and exposure bias. Dividing each patch embedding by its L2 norm:
$$\mathbf{e}_{\text{norm}} = \frac{\mathbf{e}}{\|\mathbf{e}\|_2 + \epsilon}$$
completely strips out hardware-specific illumination variance while preserving 100% of underlying geometric and biological morphology!


In [ ]:
import torch

def patch_l2_normalization(feature_map, eps=1e-8):
    """Camera-invariant patch normalization across ViT tokens."""
    norm = torch.norm(feature_map, p=2, dim=-1, keepdim=True)
    return feature_map / (norm + eps)

x = torch.randn(2, 196, 384) * 5.0 + 10.0 # simulated sensor exposure bias
x_norm = patch_l2_normalization(x)
print(f"Original patch norms: {x[0, 0].norm():.2f} -> L2 normalized patch norm: {x_norm[0, 0].norm():.4f}")
